In [4]:
# ===== Notebook 01: Randomization and Baseline =====
from src.randomization import stratified_block_randomization
from src.simulate import simulate_baseline, merge_with_randomization
import pandas as pd
from scipy import stats

# Step 1: Randomize
random_df = stratified_block_randomization(seed=1128)
print("Total subjects:", len(random_df))
print("\nGroup counts:")
print(random_df['GroupLabel'].value_counts().sort_index())

# Step 2: Simulate baseline
data1 = simulate_baseline(random_df, seed=1128)
data2 = merge_with_randomization(data1, random_df)
print("\nDATA1 shape:", data1.shape)
print("DATA2 shape:", data2.shape)

# Step 3: Balance check
def balance_test(df):
    num_vars = ['Age', 'Height', 'Weight', 'BMI', 'COPD_Duration',
                'FEV1', 'FEV1_pred_pct', 'FVC', 'FEV1_FVC']
    cat_vars = ['Sex', 'SmokingHistory', 'ConcomitantICS']
    results = []
    groups = [g for g, _ in df.groupby('Group')]
    for v in num_vars:
        arrays = [df.loc[df['Group'] == g, v].values for g in groups]
        f, p = stats.f_oneway(*arrays)
        results.append({'Variable': v, 'Test': 'ANOVA',
                        'Statistic': round(f, 3), 'p_value': round(p, 4)})
    for v in cat_vars:
        ct = pd.crosstab(df['Group'], df[v])
        chi2, p, _, _ = stats.chi2_contingency(ct)
        results.append({'Variable': v, 'Test': 'Chi-square',
                        'Statistic': round(chi2, 3), 'p_value': round(p, 4)})
    return pd.DataFrame(results)

print("\nBalance test:")
display(balance_test(data2))

# Step 4: Save outputs
random_df.to_csv('outputs/tables/randomization_table.csv', index=False)
data1.to_csv('outputs/tables/DATA1_baseline.csv', index=False)
data2.to_csv('outputs/tables/DATA2_grouped.csv', index=False)
print("\nOutputs saved.")

Total subjects: 840

Group counts:
GroupLabel
Formoterol        120
Indacaterol150    120
Indacaterol300    120
Indacaterol600    120
Indacaterol75     120
Placebo           120
Tiotropium        120
Name: count, dtype: int64

DATA1 shape: (840, 13)
DATA2 shape: (840, 19)

Balance test:


,Variable,Test,Statistic,p_value
0,Age,ANOVA,0.219,0.9708
1,Height,ANOVA,0.108,0.9956
2,Weight,ANOVA,0.311,0.9312
3,BMI,ANOVA,0.397,0.8809
4,COPD_Duration,ANOVA,0.161,0.9867
5,FEV1,ANOVA,0.154,0.9884
6,FEV1_pred_pct,ANOVA,0.175,0.9836
7,FVC,ANOVA,0.051,0.9994
8,FEV1_FVC,ANOVA,0.164,0.9862
9,Sex,Chi-square,1.090,0.9820



Outputs saved.
